# Qwen3-VL Mini

Text decoder + tiny ViT tower + projector.

## Setup

In [1]:
from pathlib import Path
import sys

PACKAGES = Path.cwd()
while not (PACKAGES / "qwen_common").exists():
    if PACKAGES == PACKAGES.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    PACKAGES = PACKAGES.parent
sys.path.insert(0, str(PACKAGES))

import torch

torch.manual_seed(0)

## Text only

In [2]:
from qwen3_vl_mini import Qwen3VLMini, preset

cfg = preset("demo")
model = Qwen3VLMini(cfg)
tokens = torch.randint(0, cfg.vocab_size, (1, 5))
text_logits, _ = model(tokens)
print("text logits:", tuple(text_logits.shape))

text logits: (1, 5, 64)


## With an image

Image tokens are prepended to the text tokens.

In [3]:
images = torch.randn(1, 3, cfg.image_size, cfg.image_size)
log = []
logits, _ = model(tokens, images=images, log=log)
n_patches = (cfg.image_size // cfg.patch_size) ** 2
print("image patches:", n_patches)
print("logits       :", tuple(logits.shape))
print("fusion log   :", log[:2])

image patches: 16
logits       : (1, 21, 64)
fusion log   : ['VL: 16 image tokens + 5 text tokens', 'QwenMini T=21 layers=2 moe=False qk_norm=True']


## Checkpoint

- What does the patch convolution do?
- Why is the output longer than the text input?